In [1]:
%pip install openai sae-lens transformer_lens einops jaxtyping circuitsvis sae_vis

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias
import transformer_lens
import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
import re
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])




In [4]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.6.hook_resid_pre",
    device=str(device),
)

Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
gpt2.generate("The Fifa world cup 2026 winner between argentina and spain is going to be", max_new_tokens = 30)

  0%|          | 0/30 [00:00<?, ?it/s]

'The Fifa world cup 2026 winner between argentina and spain is going to be announced on Thursday May 15, 2015. However, Watchdog.com revealed on Tuesday that South Africa will face Uruguay, Slovakia and Iceland on a stage'

I initially thought of going with sentiments but chose not to due to the world cup fever. I thought it would just be a better idea for my blog. I chose layer 6 because it was the only the only layer with the world cup layer present in it



In [6]:
latent_idx = 7392


In [7]:
pos_list = [
    "the world cup final is on a sunday.", "he scored a hattrick in the world cup final.",
    "the world cup lasts about five weeks.", "Argentina and Spain reached the world cup final.",
    "fans filled the stadium for the world cup.", "England advanced to the world cup semifinal.",
    "he was the standout player at this world cup.", "the world cup draw was announced today.",
    "this is his final world cup appearance.", "he is refereeing the world cup final.",
    "the world cup final is in New York.", "tickets for this world cup are expensive.",
    "that was the match of the world cup.", "the world cup is football's biggest tournament.",
    "he leads the world cup in assists.", "I played in the previous world cup.",
    "he played as a defender this world cup.", "Cape Verde had a great run this world cup.",
    "the world cup happens every four years.", "the 2026 world cup has forty-eight teams.",
    "Brazil qualified for the world#in.", "the host nation opened the world cup with a win.",
    "Croatia stunned everyone at the last world cup.", "the world cup group stage begins on friday.",
    "penalty shootouts decided the world cup quarterfinal.", "Messi lifted the world cup trophy in Qatar.",
    "the world cup qualifiers run for two years.", "Germany crashed out of the world cup early.",
    "a late goal sent them into the world cup final.", "the world cup mascot was revealed yesterday.",
    "millions watched the world cup opening ceremony.", "Morocco reached the world cup semifinals.",
    "the striker won the world cup golden boot.", "extra time was needed in the world cup final.",
    "the world cup will be hosted across three countries.", "VAR overturned a goal in the world cup match.",
    "France defended their world cup title poorly.", "the world cup final drew a billion viewers.",
    "he captained his country at three world cups.", "the world cup bracket is set for the knockouts.",
    "Japan beat Spain in the world cup group stage.", "the world cup semifinal went to penalties.",
    "their world cup campaign ended in the round of sixteen.", "the world cup anthem played before kickoff.",
    "Ronaldo missed the world cup through injury.", "the world cup expanded to forty-eight teams.",
    "a hat-trick knocked them out of the world cup.", "the world cup final whistle blew at last.",
    "fans celebrated the world cup victory all night.", "the world cup returns to North America in 2026.",
]

neutral_list = [
    "I ate lunch a bit late today.", "she painted the bedroom wall blue.",
    "I ordered some vegetables today.", "I forgot to sign the register today.",
    "I have a music class this evening.", "he sings quite well.",
    "they are jogging around the block.", "you are not going to the market today.",
    "the cafe was really crowded.", "the library closed at eight.",
    "my hostel has a curfew.", "this is a nice city to live in.",
    "I like the smell of rain.", "today is a sunny day.",
    "he sits in the park every morning.", "this is the tallest building in town.",
    "I am reading a long novel.", "the flowers on the balcony bloomed.",
    "this is a brand new dress.", "that is the biggest dog I have seen.",
    "the kettle is boiling in the kitchen.", "she takes the bus to work.",
    "we repainted the fence last weekend.", "the bakery smells wonderful in the morning.",
    "he fixed the leaking tap himself.", "the train was ten minutes late.",
    "I need to buy some groceries later.", "the meeting got moved to thursday.",
    "her cat sleeps on the windowsill.", "the printer ran out of ink again.",
    "we planted tomatoes in the garden.", "the elevator is out of order.",
    "I misplaced my house keys this morning.", "the soup needs a little more salt.",
    "they watched a documentary about whales.", "the coffee here is surprisingly good.",
    "he walks his dog every evening.", "the bookstore is having a small sale.",
    "my phone battery died at noon.", "she learned to knit last winter.",
    "the park bench was freshly painted.", "we ran out of milk this morning.",
    "the neighbours are renovating their kitchen.", "I left my umbrella on the bus.",
    "the museum opens at ten tomorrow.", "he prefers tea over coffee.",
    "the streetlights came on at dusk.", "she organized her bookshelf by color.",
    "the recipe calls for two eggs.", "I watered the plants before leaving.",
]

statements = pos_list + neutral_list
    

Just cooked up some random sentences, tried put world cup in different positions so the model doesn't cheat and instead actually target the feature direction.

In [8]:
gpt2.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': 8.0,
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='cuda'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': 0.02886751345948129,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 84934656,
 'normalization_type': 'LNPre',
 

In [9]:
gpt2_sae.cfg

StandardSAEConfig(d_in=768, d_sae=24576, dtype='float32', device='cuda', apply_b_dec_to_input=True, normalize_activations='none', reshape_activations='none', metadata=SAEMetadata({'sae_lens_version': '6.46.1', 'sae_lens_training_version': None, 'model_name': 'gpt2-small', 'hook_name': 'blocks.6.hook_resid_pre', 'hook_head_index': None, 'dataset_path': 'Skylion007/openwebtext', 'context_size': 128, 'model_from_pretrained_kwargs': {'center_writing_weights': True}, 'neuronpedia_id': 'gpt2-small/6-res-jb', 'prepend_bos': True}))

In [10]:
def diff_of_means(
    model,
    statements,
    hook_name    
):
    pos_acts= t.empty(len(pos_list), model.cfg.d_model).to(device)
    neutral_acts = t.empty(len(neutral_list),model.cfg.d_model).to(device)
    for i in range(len(statements)):
        if i<len(pos_list):
            _,cache = model.run_with_cache(statements[i])
            pos_acts[i] = cache[hook_name][0,-1]
        else:
            _,cache = model.run_with_cache(statements[i])
            neutral_acts[i-len(pos_list)] = cache[hook_name][0,-1]
    diff_of_means  = pos_acts.mean(0) - neutral_acts.mean(0)
    return diff_of_means
    


In [11]:
diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre").shape 

torch.Size([768])

In [12]:
diff_of_means(gpt2,statements,"blocks.6.hook_resid_pre").norm()

tensor(15.1288, device='cuda:0')

diff_of_means vector norm is 15.1288

In [13]:
def steering_hook(acts,hook, sae, latent_idx,steering_vector = None, steering_coefficient = 1.0):
    if steering_vector is not None:
        return acts + steering_coefficient*steering_vector
    else:
        return acts + steering_coefficient*sae.W_dec[latent_idx]

just generalised the above function

In [14]:
GENERATE_KWARGS = dict(temperature = 0.7, freq_penalty = 1.0, verbose = False)

def generate_with_steering(model, sae,prompt, latent_idx,steering_vector,steering_coefficient = 1.0, max_new_tokens = 64):
    _steering_hook = partial(steering_hook, sae = sae, latent_idx = latent_idx, steering_vector = steering_vector,steering_coefficient = steering_coefficient)
    with model.hooks(fwd_hooks = [(sae.cfg.metadata.hook_name, _steering_hook)]):
        output = model.generate(prompt,max_new_tokens = max_new_tokens,**GENERATE_KWARGS)
    return output

In [15]:
gpt2.generate(statements[21])


  0%|          | 0/10 [00:00<?, ?it/s]

'the host nation opened the world cup with a win.\n\nclare.malone@latimes'

why is the world cup steer not working properly, I lowered the freq penalty from 2.0 to 1.0, increased the max_tokens. Let's try increasing the temperature.

this was just testing on coefficient to check whether the steering is actually working or not. Now I'm going to do a sweep on a bunch of steering_coefficients just to find the best one. for that I need to create a keyword list

# Effect Metric
Just the score of the sae with fixed decoder vector and sae with a designated steering vector generated using diff_of_means. It just checks the frequency of the words present in the keyword_set in the output string generated. For example, A string like world cup qualifier cup cup cup cup cup, would have a high score comapred to a coherent sentence like I like watching the world cup.

In [31]:
keywords_set = set(['world','cup','tournament','soccer','football','qualify','qualification','knockout','match','goal','striker','score', 'championship','players','fifa','team','qualifier','qualifying','finals','knockouts'])
len(keywords_set)

20

Creating 5 fresh eval neutral statements so that I can them through during eval

In [32]:
eval_neut_statements = [
    "She goes to the gym every day.", "I like drinking tea in the morning.",
    "My neighbour seems friendly.", "My cousin used to play chess.",
    "The lights here are dim.", "He opened the window and looked out.",
    "We went for a walk after dinner.", "The weather has been strange lately.",
    "I spent the afternoon cleaning.", "She was humming an old tune.",
    "The room felt warm and quiet.", "I picked up a book from the shelf.",
    "They talked for hours over coffee.", "My favourite part of the day is evening.",
    "He parked the car near the gate.", "The kettle whistled in the kitchen.",
    "I wrote a letter to an old friend.", "The bus was almost empty tonight.",
    "She smiled and waved goodbye.", "The garden looked lovely in spring.",
    "I could not fall asleep last night.", "He tied his shoes and stepped out.",
    "The bakery was closing for the day.", "We sat by the river for a while.",
    "Nothing much happened this week.",
]

In [33]:
steering_vector = diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre")
coeff_range_diff_of_means = t.linspace(0.2,15,30)
score_dict_diff_of_means = {}

for i in coeff_range_diff_of_means:
    score = 0
    for statement in eval_neut_statements:
        output = generate_with_steering(gpt2, gpt2_sae, statement, latent_idx = 7392, steering_vector = steering_vector, steering_coefficient = i)
        tokens = re.findall('[a-z]+', output.lower())
        score += sum(tok in keywords_set for tok in tokens)/len(tokens)
    score/=len(eval_neut_statements)
    score_dict_diff_of_means[i] = score

score_dict_diff_of_means

{tensor(0.2000): 0.0007272727272727272,
 tensor(0.7103): 0.006602629676097076,
 tensor(1.2207): 0.0076606149099678,
 tensor(1.7310): 0.021082002940889962,
 tensor(2.2414): 0.02302044671950981,
 tensor(2.7517): 0.037131453350914535,
 tensor(3.2621): 0.045225610613833955,
 tensor(3.7724): 0.047134561449118914,
 tensor(4.2828): 0.05468146575444854,
 tensor(4.7931): 0.054153373202481744,
 tensor(5.3034): 0.05744385390874674,
 tensor(5.8138): 0.06181971256289108,
 tensor(6.3241): 0.0639616301633633,
 tensor(6.8345): 0.09193218802503948,
 tensor(7.3448): 0.0957503465364235,
 tensor(7.8552): 0.10481349484289948,
 tensor(8.3655): 0.1132271346919506,
 tensor(8.8759): 0.11332734837181607,
 tensor(9.3862): 0.12030608420624192,
 tensor(9.8966): 0.1182206261809585,
 tensor(10.4069): 0.09658055920405567,
 tensor(10.9172): 0.09287266230744055,
 tensor(11.4276): 0.07809028633307523,
 tensor(11.9379): 0.08745110121104017,
 tensor(12.4483): 0.09465732684460607,
 tensor(12.9586): 0.0837352326258123,
 ten

In [34]:
coeff_range_plain = t.linspace(15,100,18)
score_dict_plain = {}
for i in coeff_range_plain:
    score = 0
    for statement in eval_neut_statements:
        output = generate_with_steering(gpt2, gpt2_sae, statement, latent_idx = 7392,steering_vector = None,steering_coefficient = i)
        tokens = re.findall('[a-z]+',output.lower())
        score = sum(tok in keywords_set for tok in tokens)/len(tokens)
    score/=len(eval_neut_statements)
    score_dict_plain[i] = score
score_dict_plain

{tensor(15.): 0.0,
 tensor(20.): 0.0013559322033898306,
 tensor(25.): 0.0032786885245901635,
 tensor(30.): 0.0032786885245901635,
 tensor(35.): 0.0019672131147540984,
 tensor(40.): 0.00125,
 tensor(45.): 0.002962962962962963,
 tensor(50.): 0.002807017543859649,
 tensor(55.): 0.0015384615384615385,
 tensor(60.): 0.0036363636363636364,
 tensor(65.): 0.0044444444444444444,
 tensor(70.): 0.0044444444444444444,
 tensor(75.): 0.004,
 tensor(80.): 0.004137931034482759,
 tensor(85.): 0.005,
 tensor(90.): 0.003773584905660378,
 tensor(95.): 0.004615384615384616,
 tensor(100.): 0.00816326530612245}

In [35]:
max(score_dict_diff_of_means.items(), key = lambda x: x[1]), max(score_dict_plain.items(), key = lambda x : x[1])

((tensor(9.3862), 0.12030608420624192), (tensor(100.), 0.00816326530612245))

# Coherence (Perplexity)
I think effect metric is done, in the meantime let's just ponder upon coding the coherence function. What is the conherence function, it is just how interpretable and coherent the sentence is. It checks wheter ths sentence is pure gibberish or an actual grammatically correct sentence.

In [36]:
def perplexity(model, prompt):
    loss = model(prompt, return_type = "loss")
    perplexity = t.exp(loss)
    return perplexity.item()

In [57]:
steering_vector = diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre")
coeff_range_diff_of_means = t.linspace(0.2,15,30)
ppl_dict_diff_of_means = {}

for i in coeff_range_diff_of_means:
    ppl = 0
    for statement in eval_neut_statements:
        output = generate_with_steering(gpt2, gpt2_sae, statement, latent_idx = 7392, steering_vector = steering_vector, steering_coefficient = i)
        ppl+= perplexity(gpt2, output)
    ppl/=len(eval_neut_statements)
    ppl_dict_diff_of_means[i] = ppl

ppl_dict_diff_of_means

{tensor(0.2000): 14.147955131530761,
 tensor(0.7103): 14.489163513183593,
 tensor(1.2207): 17.182433547973634,
 tensor(1.7310): 21.96603889465332,
 tensor(2.2414): 26.729848976135255,
 tensor(2.7517): 32.95278076171875,
 tensor(3.2621): 49.021278076171875,
 tensor(3.7724): 55.659350891113284,
 tensor(4.2828): 70.39745697021485,
 tensor(4.7931): 88.48207397460938,
 tensor(5.3034): 94.22368606567383,
 tensor(5.8138): 124.52700897216796,
 tensor(6.3241): 139.84036712646486,
 tensor(6.8345): 210.3444500732422,
 tensor(7.3448): 276.71162170410156,
 tensor(7.8552): 285.66681030273435,
 tensor(8.3655): 336.61718902587893,
 tensor(8.8759): 343.9606201171875,
 tensor(9.3862): 385.69023193359374,
 tensor(9.8966): 410.2195080566406,
 tensor(10.4069): 551.2342517089844,
 tensor(10.9172): 477.75438720703124,
 tensor(11.4276): 513.8651315307617,
 tensor(11.9379): 567.9266198730469,
 tensor(12.4483): 574.6952380371093,
 tensor(12.9586): 575.9626306915284,
 tensor(13.4690): 725.0665747070312,
 tensor(

In [38]:
steering_vector = diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre")
coeff_range_plain = t.linspace(15,100,18)
ppl_dict_plain = {}
for i in coeff_range_plain:
    ppl = 0
    for statement in eval_neut_statements:
        output = generate_with_steering(gpt2, gpt2_sae, statement, latent_idx = 7392,steering_vector = None,steering_coefficient = i)
        ppl += perplexity(gpt2,output)
    ppl/=len(eval_neut_statements)
    ppl_dict_plain[i] = ppl
ppl_dict_plain

{tensor(15.): 17.462779922485353,
 tensor(20.): 16.706762771606446,
 tensor(25.): 21.665450286865234,
 tensor(30.): 23.118427276611328,
 tensor(35.): 31.914979782104492,
 tensor(40.): 39.83458221435547,
 tensor(45.): 44.7624560546875,
 tensor(50.): 71.57342529296875,
 tensor(55.): 95.91085029602051,
 tensor(60.): 129.11719665527343,
 tensor(65.): 152.83490600585938,
 tensor(70.): 251.3218899536133,
 tensor(75.): 383.45464111328124,
 tensor(80.): 401.5836627197266,
 tensor(85.): 460.474052734375,
 tensor(90.): 582.4944689941407,
 tensor(95.): 696.7575721740723,
 tensor(100.): 719.5732092285157}

In [58]:
min(ppl_dict_diff_of_means.items(), key = lambda x:x[1]), min(ppl_dict_plain.items(), key = lambda x: x[1])

((tensor(0.2000), 14.147955131530761), (tensor(20.), 16.706762771606446))

The lower the perplexity the better as it shows that the model is less confused between what words to choose from. Lower perplexity means higher coherence of sentences. Now we are going to plot perplexity vs effect to find the best possible steering_coefficient

In [59]:
def frontier_xy(effect_dict, ppl_dict):
    eff = sorted(effect_dict.items(), key=lambda kv: float(kv[0]))
    ppl = sorted(ppl_dict.items(),    key=lambda kv: float(kv[0]))
    x = [float(v) for _, v in ppl]     # perplexity (x)
    y = [float(v) for _, v in eff]     # effect     (y)
    return x, y

x_diff, y_diff = frontier_xy(score_dict_diff_of_means, ppl_dict_diff_of_means)
x_sae,  y_sae  = frontier_xy(score_dict_plain,         ppl_dict_plain)

fig = go.Figure()
fig.add_scatter(x=x_diff, y=y_diff, mode='lines+markers', name='diff-of-means')
fig.add_scatter(x=x_sae,  y=y_sae,  mode='lines+markers', name='SAE (naive)')
fig.update_layout(
    title='Steering frontier: World Cup effect vs coherence',
    xaxis_title='perplexity  (lower = more coherent)',
    yaxis_title='World Cup effect  (higher = stronger)',
    legend_title='method',
)
fig.show()

SO the diff_of_means completely destroyed the plain sae. This is probably due to diff_of_means being a supervised vector trained on proper labels while sae.W_enc[latent_idx] being a completely unsupervised one.

# Steelman SAE
The best way to stay steer an SAE

In [41]:
def max_activation(model, sae,hook_name,latent_idx,pos_list):
   max_list = []
   for i in pos_list:
        _,cache = model.run_with_cache(i)
        feats = sae.encode(cache[hook_name])
        max_act = feats[...,latent_idx].max()
        max_list.append(max_act.item())
   return max(max_list)

max_act = max_activation(gpt2,gpt2_sae, "blocks.6.hook_resid_pre",latent_idx, pos_list)

so I the max_act of my sae is 24.53 but on neuronpedia it was showing 52. The discrepancy could just be due to the different sizes of the datasets. Now let's move on to actually clamping the encode block, the core of the steelman. In this we equate the feats we obtain after the encode block to a specified target for that particular latent_idx. I am currently going to perform across the targets to find the best one, the ideal estimate for the target is said to be about 1.5 or 2x the max_act.

In [42]:
def steelman_hook(acts, hook, sae, latent_idx, target):
    feats = sae.encode(acts)
    error = acts - sae.decode(feats) 
    feats[..., latent_idx] = target
    return sae.decode(feats) + error

In [43]:
def generate_with_steelman(model,sae,statement,latent_idx,target,max_new_tokens = 64):
    _steelman_hook = partial(steelman_hook, sae = sae, latent_idx = latent_idx, target = target)
    with model.hooks(fwd_hooks = [(sae.cfg.metadata.hook_name, _steelman_hook)]):
        output = model.generate(statement,max_new_tokens = max_new_tokens,**GENERATE_KWARGS)
    return output

In [44]:
def steelman_hook_decay(acts, hook, sae, latent_idx, prompt_len ,target,rate = 0.98):
    feats = sae.encode(acts)
    error = acts - sae.decode(feats) 
    pos = t.arange(feats.shape[1],device = device)
    offset = (pos - prompt_len).clamp(min = 0.0)
    schedule = t.where(pos>=prompt_len, target*(rate**offset),0.0)
    feats[..., latent_idx] = schedule
    return sae.decode(feats) + error

In [45]:
GENERATE_KWARGS = dict(temperature = 0.7, freq_penalty = 1.0, verbose = False,use_past_kv_cache = False)
def generate_with_steelman_decay(model,sae,statement,latent_idx,target,max_new_tokens = 64):
    prompt_len = model.to_tokens(statement).shape[-1]
    _steelman_hook = partial(steelman_hook_decay, sae = sae, latent_idx = latent_idx, prompt_len = prompt_len, target = target)
    with model.hooks(fwd_hooks = [(sae.cfg.metadata.hook_name, _steelman_hook)]):
        output = model.generate(statement,max_new_tokens = max_new_tokens,**GENERATE_KWARGS)
    return output

In [46]:
targets = t.linspace(0, 8*max_act, 20)
score_dict_steelman = {}

for i in targets:
    score = 0
    for statement in eval_neut_statements:
        output = generate_with_steelman(gpt2, gpt2_sae, statement, latent_idx = 7392, target = i)
        tokens = re.findall('[a-z]+', output.lower())
        score += sum(tok in keywords_set for tok in tokens)/len(tokens)
    score/=len(eval_neut_statements)
    score_dict_steelman[i] = score

score_dict_steelman

{tensor(0.): 0.0007017543859649122,
 tensor(11.1914): 0.005928092501368363,
 tensor(22.3827): 0.023820986624190087,
 tensor(33.5741): 0.03278689223805132,
 tensor(44.7655): 0.04833129262411195,
 tensor(55.9568): 0.07041558187897025,
 tensor(67.1482): 0.12355099708652537,
 tensor(78.3396): 0.14134021331713703,
 tensor(89.5309): 0.16026247199009372,
 tensor(100.7223): 0.16426598057181008,
 tensor(111.9137): 0.1941278480559998,
 tensor(123.1050): 0.1969078783334531,
 tensor(134.2964): 0.21500463779373422,
 tensor(145.4877): 0.22005099877657827,
 tensor(156.6791): 0.24135974203129273,
 tensor(167.8705): 0.2490537117459688,
 tensor(179.0618): 0.25819896200547776,
 tensor(190.2532): 0.2649472326277045,
 tensor(201.4446): 0.2737745461866475,
 tensor(212.6359): 0.27782109115112585}

In [47]:
ppl_dict_steelman = {}
for i in targets:
    ppl = 0
    for statement in eval_neut_statements:
        output = generate_with_steelman(gpt2, gpt2_sae, statement, latent_idx = 7392, target = i)
        ppl += perplexity(gpt2,output)
    ppl/=len(eval_neut_statements)
    ppl_dict_steelman[i] = ppl
ppl_dict_steelman

{tensor(0.): 15.209470787048339,
 tensor(11.1914): 14.211244506835937,
 tensor(22.3827): 16.687416629791258,
 tensor(33.5741): 26.99116226196289,
 tensor(44.7655): 46.06077583312988,
 tensor(55.9568): 78.9930110168457,
 tensor(67.1482): 215.26982666015624,
 tensor(78.3396): 413.8774169921875,
 tensor(89.5309): 662.9279113769531,
 tensor(100.7223): 821.6285778808593,
 tensor(111.9137): 808.6960546875,
 tensor(123.1050): 783.0569384765626,
 tensor(134.2964): 912.620419921875,
 tensor(145.4877): 844.8437316894531,
 tensor(156.6791): 766.8844958496094,
 tensor(167.8705): 807.5192224121093,
 tensor(179.0618): 663.4460961914062,
 tensor(190.2532): 744.2561926269532,
 tensor(201.4446): 686.8059887695313,
 tensor(212.6359): 620.4023706054687}

In [48]:
targets = t.linspace(0, 8*max_act, 20)
score_dict_steelman_decay = {}

for i in targets:
    score = 0
    for statement in eval_neut_statements:
        output = generate_with_steelman_decay(gpt2, gpt2_sae, statement, latent_idx = 7392, target = i)
        tokens = re.findall('[a-z]+', output.lower())
        score += sum(tok in keywords_set for tok in tokens)/len(tokens)
    score/=len(eval_neut_statements)
    score_dict_steelman_decay[i] = score

score_dict_steelman_decay

{tensor(0.): 0.003070175438596491,
 tensor(11.1914): 0.003502304147465438,
 tensor(22.3827): 0.007500312484976683,
 tensor(33.5741): 0.012394993460676042,
 tensor(44.7655): 0.02758167412880324,
 tensor(55.9568): 0.026628360695125437,
 tensor(67.1482): 0.02969234748082482,
 tensor(78.3396): 0.03587108761970203,
 tensor(89.5309): 0.06091962999719958,
 tensor(100.7223): 0.07261147172815113,
 tensor(111.9137): 0.07618341538315435,
 tensor(123.1050): 0.08966117246955463,
 tensor(134.2964): 0.08084869188507068,
 tensor(145.4877): 0.11507758027224227,
 tensor(156.6791): 0.11022072504252613,
 tensor(167.8705): 0.12053312483782701,
 tensor(179.0618): 0.12830591700263402,
 tensor(190.2532): 0.14182203059497397,
 tensor(201.4446): 0.13858239597203267,
 tensor(212.6359): 0.16199872724960226}

In [49]:
ppl_dict_steelman_decay = {}
for i in targets:
    ppl = 0
    for statement in eval_neut_statements:
        output = generate_with_steelman_decay(gpt2, gpt2_sae, statement, latent_idx = 7392, target = i)
        ppl += perplexity(gpt2,output)
    ppl/=len(eval_neut_statements)
    ppl_dict_steelman_decay[i] = ppl
ppl_dict_steelman_decay

{tensor(0.): 13.049979400634765,
 tensor(11.1914): 14.187976608276367,
 tensor(22.3827): 15.911150970458984,
 tensor(33.5741): 17.693397941589357,
 tensor(44.7655): 24.105239868164062,
 tensor(55.9568): 28.401963348388673,
 tensor(67.1482): 40.738234786987306,
 tensor(78.3396): 58.75908889770508,
 tensor(89.5309): 86.4831967163086,
 tensor(100.7223): 195.55052383422853,
 tensor(111.9137): 184.54963012695313,
 tensor(123.1050): 265.45418640136717,
 tensor(134.2964): 277.9431671142578,
 tensor(145.4877): 420.20466278076174,
 tensor(156.6791): 499.4838977050781,
 tensor(167.8705): 572.2444958496094,
 tensor(179.0618): 586.5211578369141,
 tensor(190.2532): 580.2119750976562,
 tensor(201.4446): 766.6352514648438,
 tensor(212.6359): 767.3772216796875}

In [70]:
x_diff, y_diff = frontier_xy(score_dict_diff_of_means, ppl_dict_diff_of_means)
x_sae,  y_sae  = frontier_xy(score_dict_plain,ppl_dict_plain)
x_steel, y_steel = frontier_xy(score_dict_steelman, ppl_dict_steelman)
x_steel_decay, y_steel_decay = frontier_xy(score_dict_steelman_decay,ppl_dict_steelman_decay)

fig = go.Figure()
fig.add_scatter(x=x_diff, y=y_diff, mode='lines+markers', name='diff-of-means')
fig.add_scatter(x=x_sae,  y=y_sae,  mode='lines+markers', name='SAE (naive)')
fig.add_scatter(x=x_steel, y=y_steel, mode='lines+markers', name='SAE (steelman)')
fig.add_scatter(x = x_steel_decay,y = y_steel_decay,mode = 'lines+markers',name = 'SAE (steelman with decay)')
fig.update_layout(
    title='Steering frontier: World Cup effect vs coherence',
    xaxis_title='perplexity  (lower = more cohe/rent)',
    yaxis_title='World Cup effect  (higher = stronger)',
    legend_title='method',
)
fig.update_xaxes(range = [0,150])
fig.show()



# Sentence Demonstrations

**Examples of sentences at high perplexities**


In [61]:
generate_with_steelman(gpt2,gpt2_sae,eval_neut_statements[0],latent_idx = 7392,target = 55.9568)

'She goes to the gym every day. Women, and they are being not qualifiers in qualification for, qualifying for, supported team qualifiers. In Brazil and qualifying for, we should be supporting a more local qualifier in 2022."\n\nThe stay of bronze-winning hopeful is not yet over\'s looming competition in South Korea next summer and includes Germany\'s group of qualifiers'

In [73]:
generate_with_steelman_decay(gpt2,gpt2_sae,eval_neut_statements[0],latent_idx = 7392, target = 89.5309)

"She goes to the gym every day. The after that. and, I in is qualifying for helping countries around qualifier qualifiers in the qualifiers. With qualifying looming, she has two to go and we should have one with Brazil next year in Brazil on February 13 and then South Africa if it's not there so they can qualify there but no action. It's always"

In [74]:
steering_vector = diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre")

generate_with_steering(gpt2, gpt2_sae, eval_neut_statements[0], latent_idx = 7392, steering_vector = steering_vector, steering_coefficient = 4.7931)

"She goes to the gym every day. They are all players through qualifying points and men's sledge and medal winners should have been in a series of rounds which was the first of six-12 is now Europe, the US, who are 14 and 12 winning men's qualification series.\n\nThe 18-16 were played in Russia and European final and will"

In [76]:
generate_with_steering(gpt2, gpt2_sae, eval_neut_statements[0], latent_idx = 7392, steering_vector = None, steering_coefficient = 55)

"She goes to the gym every day.\n\nA quarter-final qualifier for Ireland in 2012, in Brazil is not tournament qualifying but a qualifier that is never to be played.\n\nHer two competitors in Brazil are not yet ready and qualifying campaign plans didn't go as well. They are football qualifiers, qualifiers or even World Cup finals? qualifiers. In"

**Examples of sentences at low perplexities**

In [69]:
generate_with_steelman(gpt2, gpt2_sae, eval_neut_statements[0], latent_idx = 7392, target = 33.5741 )

"She goes to the gym every day.\n\nThis year, in her first tournament, she will be unable to finish any spells against Wales and Portugal at home but she's not sure if she can get that up next time.\n\nHer last-team experience would make her even better but she is one first-class performer ready when England kicks off in"

In [67]:
steering_vector = diff_of_means(gpt2, statements, "blocks.6.hook_resid_pre")

generate_with_steering(gpt2, gpt2_sae, eval_neut_statements[0], latent_idx = 7392, steering_vector = steering_vector, steering_coefficient = 2.2414)

'She goes to the gym every day. That her body is to be admired.\n\n"She\'s right there," she said. "All that does it."'

In [68]:
generate_with_steelman_decay(gpt2,gpt2_sae,eval_neut_statements[0],latent_idx = 7392, target = 44.7655)

"She goes to the gym every day. She just thinks of her past and now has a deal to take the blame.\n\nShe was wiped out by Spain in 2007\n\nIn 2007, she made it back on her feet after being ruled out for five years by England's first-choice squad. Her fans were disappointed, but so were England's."

In [71]:
generate_with_steering(gpt2, gpt2_sae, eval_neut_statements[0], latent_idx = 7392, steering_vector = None, steering_coefficient = 25)

"She goes to the gym every day. No, she's not going to be playing in the Olympic training camp in Brazil, or qualifying for the 2014 World Cup in Russia, or any other international.\n\nShe's not going to be a star player. But if she is, it will be because of that. Can you imagine what it would look like"